# Euler 3-Wheeler — SoH Forecast Model

Condition-aware **State-of-Health forecasting** for the Euler electric-3W cohort, with
**P10–P90 uncertainty bands**, validated by a **leave-one-vehicle-out (LOVO) backtest**.

- **Target**: validated BMS remaining-capacity SoH (high-SoC, isotonic) — `data/euler/features/feature_table.parquet`.
- **Model** (`src/euler_model.py`): a *trajectory* forecaster — pooled LightGBM that predicts cumulative
  SoH loss from an anchor month as a stress-conditioned √-age curve, **blended with a continuation of each
  vehicle's own recent slope** (so fast decliners keep declining and flat vehicles stay flat), plus a
  **calibrated empirical band** for P10/P90.
- **Validation**: LOVO (train on the other vehicles, forecast each vehicle's held-out last ~40%), split
  into **degrading (tail ≥ 2 pp) vs flat**, vs **persistence**, **√t trend**, the legacy **rate model**, and
  the **Chronos** univariate foundation model.

> Small fleet (~22 vehicles): we keep the simplest thing that validates and are explicit about where the
> model wins and loses.


In [ ]:
# --- repo-root cwd + imports ---
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

import euler_model as em
import euler_backtest as bt

TEMPLATE = 'plotly_white'
PAL = px.colors.qualitative.Safe
EOL = em.EOL                 # 80% end-of-first-life
WARRANTY_YEARS = 5.0         # config.warranty_for('euler','') -> 5 yr

m = pd.read_parquet('data/euler/features/feature_table.parquet')
m['month'] = pd.to_datetime(m['month'])
m = m.sort_values(['vin', 'month']).reset_index(drop=True)
print(f'{m.vin.nunique()} vehicles, {len(m)} vin-month rows, '
      f"SoH range {m.soh.min():.0f}-{m.soh.max():.0f}")
m.head(3)


## 1. Cohort overview — observed SoH trajectories
Each line is one vehicle's BMS-capacity SoH vs age. Colour = total observed drop (red = heavy degrader).

In [ ]:
drops = (m.groupby('vin')['soh'].first() - m.groupby('vin')['soh'].last())
fig = go.Figure()
for vin, g in m.groupby('vin'):
    g = g.sort_values('age_months')
    d = float(drops[vin])
    fig.add_scatter(x=g['age_months'], y=g['soh'], mode='lines',
                    name=vin[-6:], line=dict(width=1.6,
                    color=px.colors.sample_colorscale('RdYlGn_r', min(d/24, 1.0))[0]),
                    hovertemplate=f'{vin[-6:]}<br>age %{{x:.0f}} mo<br>SoH %{{y:.1f}}%<extra></extra>')
fig.add_hline(y=EOL, line=dict(color='firebrick', dash='dot'),
              annotation_text='80% EoFL', annotation_position='bottom right')
fig.update_layout(template=TEMPLATE, height=460, title='Euler cohort — observed SoH vs age',
                  xaxis_title='age (months)', yaxis_title='SoH (%)',
                  yaxis_range=[72, 101], showlegend=True,
                  legend=dict(font=dict(size=9), title='VIN (last 6)'))
fig.show()


## 2. Leave-one-vehicle-out backtest

For every vehicle we **train on the other vehicles only**, then forecast its held-out last ~40% of
months from the earlier history. We compare the **trajectory** model against the legacy **rate** model
(gated & raw), **persistence**, a **√t trend**, and **Chronos** (univariate foundation model, optional).


In [ ]:
RUN_CHRONOS = True   # set False to skip the foundation-model baseline (slower, CPU)
try:
    per = bt.run_backtest(m, with_chronos=RUN_CHRONOS)
except Exception as e:
    print('chronos failed, retrying without:', e)
    per = bt.run_backtest(m, with_chronos=False)
per_cal, band = bt.recalibrate(per, m)      # fit the calibrated P10-P90 envelope from LOVO residuals
summary = bt.summarize(per)
calib = bt.calibration(per_cal)
print('calibrated band (pp per sqrt-month):', {k: round(v, 2) for k, v in band.items()})
summary


### 2a. Backtest comparison chart — RMSE by cohort & model
Lower is better. The trajectory model is the best general forecaster and the clear winner on **degrading** vehicles vs the baselines.

In [ ]:
model_cols = [c[:-5] for c in summary.columns if c.endswith('_RMSE')]
labels = {'trajectory':'Trajectory (ours)','rate_gated':'Rate (gated)','rate_raw':'Rate (raw)',
          'persistence':'Persistence','trend':'√t trend','chronos':'Chronos (FM)'}
fig = make_subplots(rows=1, cols=2, subplot_titles=('RMSE (SoH pp)','MAE (SoH pp)'))
for ci, metric in enumerate(['RMSE','MAE'], start=1):
    for j, mc in enumerate(model_cols):
        col = f'{mc}_{metric}'
        fig.add_bar(x=summary['cohort'], y=summary[col], name=labels.get(mc, mc),
                    marker_color=PAL[j % len(PAL)], legendgroup=mc,
                    showlegend=(ci == 1), row=1, col=ci,
                    hovertemplate=f'{labels.get(mc,mc)}<br>%{{x}}: %{{y:.2f}}<extra></extra>')
fig.update_layout(template=TEMPLATE, height=440, barmode='group',
                  title='LOVO backtest — error by cohort & model (lower is better)',
                  legend=dict(orientation='h', y=1.14))
fig.update_yaxes(title_text='SoH pp')
fig.show()


### 2b. Predicted vs actual & where the model wins / loses

Left: every held-out point, trajectory P50 vs actual (points on the diagonal = perfect).
Right: per-vehicle RMSE, trajectory vs the best baseline — bars below the line are vehicles we win.


In [ ]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=('Trajectory P50 vs actual','Per-vehicle RMSE: ours vs best baseline'))
for deg, name, c in [(True,'degrading','#d62728'),(False,'flat','#2ca02c')]:
    s = per[per.degrading == deg]
    fig.add_scatter(x=s.actual, y=s.trajectory, mode='markers', name=name,
                    marker=dict(size=6, color=c, opacity=0.6),
                    hovertemplate='actual %{x:.1f}<br>pred %{y:.1f}<extra></extra>', row=1, col=1)
lo, hi = per.actual.min()-1, 101
fig.add_scatter(x=[lo,hi], y=[lo,hi], mode='lines', line=dict(color='gray', dash='dash'),
                showlegend=False, row=1, col=1)
# per-vehicle: ours vs best of (persistence,trend)
pv = []
for vin, s in per.groupby('vin'):
    base = min(rmse(s.actual, s.persistence), rmse(s.actual, s.trend))
    pv.append((vin[-6:], rmse(s.actual, s.trajectory), base, bool(s.degrading.iloc[0])))
pv = pd.DataFrame(pv, columns=['vin','ours','baseline','deg']).sort_values('ours')
fig.add_bar(x=pv.vin, y=pv.ours, name='Trajectory (ours)', marker_color='#1f77b4',
            row=1, col=2, showlegend=True)
fig.add_scatter(x=pv.vin, y=pv.baseline, mode='markers', name='best baseline',
                marker=dict(symbol='line-ew-open', size=12, color='black', line_width=2),
                row=1, col=2)
fig.update_xaxes(title_text='actual SoH (%)', row=1, col=1)
fig.update_yaxes(title_text='predicted SoH (%)', row=1, col=1)
fig.update_xaxes(title_text='VIN (last 6)', row=1, col=2, tickangle=-60)
fig.update_yaxes(title_text='RMSE (pp)', row=1, col=2)
fig.update_layout(template=TEMPLATE, height=460, legend=dict(orientation='h', y=1.13),
                  title='Predicted vs actual, and per-vehicle accuracy')
fig.show()


### 2c. Uncertainty-band calibration

The P10–P90 band should contain ≈ **80 %** of held-out actuals. We report coverage overall and by cohort.


In [ ]:
cov = pd.DataFrame({'cohort': list(calib.keys()),
                    'coverage': [calib[k] for k in calib]})
fig = go.Figure()
fig.add_bar(x=cov.cohort, y=cov.coverage, marker_color='#4c78a8',
            text=[f'{v:.0%}' for v in cov.coverage], textposition='outside')
fig.add_hline(y=0.80, line=dict(color='firebrick', dash='dash'),
              annotation_text='target 80%', annotation_position='top left')
fig.update_layout(template=TEMPLATE, height=380, yaxis_tickformat='.0%',
                  yaxis_range=[0, 1.05], yaxis_title='actuals inside P10–P90',
                  title='P10–P90 band calibration (target ≈ 80%)')
fig.show()
print('coverage:', {k: round(v, 2) for k, v in calib.items()})


## 3. Feature importance & SHAP

What drives the forecast. We fit the trajectory model on the **full** fleet and read LightGBM gain
importance, then SHAP values (mean |SHAP|) for a model-agnostic view.


In [ ]:
samples = em.build_traj_samples(m)
full_mdl = em.train_traj(samples)
full_mdl['band'] = band                         # ship the calibrated band with the model
imp = pd.Series(full_mdl['p50'].feature_importances_, index=em.TRAJ_FEATS)
imp = (imp / imp.sum() * 100).sort_values()

fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation='h',
                       marker_color='#54a24b',
                       hovertemplate='%{y}: %{x:.1f}%<extra></extra>'))
fig.update_layout(template=TEMPLATE, height=460, xaxis_title='gain importance (%)',
                  title='Trajectory model — feature importance')
fig.show()


In [ ]:
# SHAP (mean |value|) — model-agnostic importance on a sample of rows
import shap
Xs = samples[em.TRAJ_FEATS]
expl = shap.TreeExplainer(full_mdl['p50'])
sv = expl.shap_values(Xs.sample(min(1500, len(Xs)), random_state=0))
mean_abs = pd.Series(np.abs(sv).mean(0), index=em.TRAJ_FEATS).sort_values()
fig = go.Figure(go.Bar(x=mean_abs.values, y=mean_abs.index, orientation='h',
                       marker_color='#e45756',
                       hovertemplate='%{y}: %{x:.3f}<extra></extra>'))
fig.update_layout(template=TEMPLATE, height=460, xaxis_title='mean |SHAP| (pp of cumulative loss)',
                  title='Trajectory model — SHAP feature importance')
fig.show()


## 4. Per-vehicle forecasts to the 5-year warranty horizon

Observed SoH (markers) + **P50 forecast** (line) with the **P10–P90 band** (shaded), extended to the
5-year warranty horizon, with the 80% EoFL line. We show a few of the most-degraded and a couple of flat
vehicles so the band behaviour is visible in both regimes.


In [ ]:
reg = pd.read_csv('data/euler/Euler_Regd_Details.csv')
reg['reg'] = pd.to_datetime(reg['regd_date'], format='%d/%m/%y', errors='coerce')
REG = dict(zip(reg['vin'], reg['reg']))

def forecast_to_warranty(vin):
    g = m[m.vin == vin].sort_values('month')
    last_age = float(g['age_months'].iloc[-1])
    H = max(int(round(WARRANTY_YEARS * 12 - last_age)), 1)
    q = em.forecast(g, full_mdl, H)
    fmonths = [g['month'].iloc[-1] + pd.DateOffset(months=k) for k in range(1, H + 1)]
    return g, fmonths, q

def plot_vehicle(vin, ax_fig=None, row=None, col=None):
    g, fmonths, q = forecast_to_warranty(vin)
    traces = []
    fig = ax_fig or go.Figure()
    add = (lambda **kw: fig.add_trace(go.Scatter(**kw), row=row, col=col)) if ax_fig else (lambda **kw: fig.add_scatter(**kw))
    # band
    add(x=fmonths + fmonths[::-1], y=list(q[0.9]) + list(q[0.1])[::-1], fill='toself',
        fillcolor='rgba(31,119,180,0.15)', line=dict(width=0), name='P10–P90',
        showlegend=(row in (None, 1) and col in (None, 1)), hoverinfo='skip')
    add(x=fmonths, y=q[0.5], mode='lines', line=dict(color='#1f77b4', width=2.4, dash='dash'),
        name='P50 forecast', showlegend=(row in (None,1) and col in (None,1)))
    add(x=g['month'], y=g['soh'], mode='markers+lines', marker=dict(color='#222', size=5),
        line=dict(color='rgba(0,0,0,0.35)', width=1.3), name='observed SoH',
        showlegend=(row in (None,1) and col in (None,1)))
    return fig, g, fmonths, q

# grid of 6 vehicles: 4 worst degraders + 2 flat
order = drops.sort_values(ascending=False)
pick = list(order.index[:4]) + list(order.index[-2:])
fig = make_subplots(rows=2, cols=3, subplot_titles=[v[-6:] for v in pick],
                    vertical_spacing=0.13, horizontal_spacing=0.06)
for i, vin in enumerate(pick):
    r, c = i // 3 + 1, i % 3 + 1
    _, g, fmonths, q = plot_vehicle(vin, fig, r, c)
    fig.add_hline(y=EOL, line=dict(color='firebrick', dash='dot'), row=r, col=c)
    rg = REG.get(vin)
    if pd.notna(rg):
        warr = rg + pd.DateOffset(years=int(WARRANTY_YEARS))
        fig.add_vline(x=warr, line=dict(color='seagreen', dash='dashdot'), row=r, col=c)
fig.update_layout(template=TEMPLATE, height=620, title='Per-vehicle SoH forecast to 5-yr warranty (P10–P90 band)',
                  legend=dict(orientation='h', y=1.08))
fig.update_yaxes(range=[70, 101])
fig.show()


### 4a. Single-vehicle detail — the most-degraded unit
Full-resolution view of the steepest decliner: observed history, P50 forecast, P10–P90 band, EoFL and warranty markers, and the projected SoH at the warranty date.

In [ ]:
vin = order.index[0]
fig, g, fmonths, q = plot_vehicle(vin)
fig.add_hline(y=EOL, line=dict(color='firebrick', dash='dot'),
              annotation_text='80% EoFL', annotation_position='bottom right')
rg = REG.get(vin)
if pd.notna(rg):
    warr = rg + pd.DateOffset(years=int(WARRANTY_YEARS))
    fig.add_vline(x=warr, line=dict(color='seagreen', dash='dashdot'),
                  annotation_text='5-yr warranty', annotation_position='top right')
    proj = float(np.interp(warr.value, [t.value for t in fmonths], q[0.5]))
    proj_lo = float(np.interp(warr.value, [t.value for t in fmonths], q[0.1]))
    print(f'{vin}: projected SoH @ warranty  P50={proj:.1f}%  (P10={proj_lo:.1f}%)')
fig.update_layout(template=TEMPLATE, height=460, yaxis_range=[70, 101],
                  title=f'{vin[-6:]} — SoH forecast with P10–P90 band',
                  xaxis_title='date', yaxis_title='SoH (%)',
                  legend=dict(orientation='h', y=1.12))
fig.show()


## 5. Summary

- **Model**: stress-conditioned **trajectory** forecaster (pooled LightGBM cumulative-loss as a √-age
  curve) **blended with each vehicle's own recent slope**, plus a **calibrated empirical P10–P90 band**.
  The legacy **rate model** API (`build_transitions`/`train`/`free_run`) is preserved and hardened with a
  flat-vehicle gate, but the trajectory model is the one we forecast with.
- **Validation (LOVO)**: the trajectory model is the best general forecaster — it beats persistence, the
  √t trend, the legacy rate model, and Chronos overall and on **degrading** vehicles; on genuinely **flat**
  vehicles persistence is unbeatable (they don't move) but our model no longer manufactures loss there.
- **Bands** are calibrated to ≈80% P10–P90 coverage across cohorts.
- **Caveat**: ~22 vehicles is a small fleet; the pooled model regresses the 2 steepest decliners toward
  the fleet mean (the band covers them). We keep the simplest approach that validates.
